# CogVideoX-5B — Colab T4

**Versatile video generation** with CogVideoX-5B (THUDM/ZhipuAI) on a free T4 GPU.

### Why CogVideoX?
| | CogVideoX-5B | Wan 2.2 14B | HunyuanVideo 1.5 |
|---|---|---|---|
| **Model size** | ~5 GB (INT8) | ~14 GB (FP8) | ~7.76 GB (FP8) |
| **Modes** | T2V, I2V, Video extend | T2V, I2V, V2V, Talking | T2V |
| **Style** | Smooth, cinematic | Realistic | Realistic |
| **Speed** | Fast | Slow | Medium |
| **Best for** | Stylized content, variety | Realism, complex scenes | Quick iterations |

CogVideoX produces a **different aesthetic** — smoother, more cinematic. Great for variety in your video pipeline.

**Run cells 1-6 in order**, then open the tunnel URL.

In [ ]:
#@title 1. Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
#@title 2. Install ComfyUI + Custom Nodes
%cd /content
!git clone https://github.com/comfyanonymous/ComfyUI.git 2>/dev/null || echo "Already cloned"
%cd /content/ComfyUI
!pip install -r requirements.txt -q

# Custom nodes
%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/kijai/ComfyUI-CogVideoXWrapper.git 2>/dev/null || echo "Already cloned"
!git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git 2>/dev/null || echo "Already cloned"
!git clone https://github.com/kijai/ComfyUI-KJNodes.git 2>/dev/null || echo "Already cloned"

!pip install -r ComfyUI-CogVideoXWrapper/requirements.txt -q
!pip install -r ComfyUI-VideoHelperSuite/requirements.txt -q
!pip install -r ComfyUI-KJNodes/requirements.txt -q

print("\nInstalled!")

In [ ]:
#@title 3. Download CogVideoX Models (~10 GB)
import os

M = "/content/ComfyUI/models"
os.makedirs(f"{M}/diffusion_models", exist_ok=True)
os.makedirs(f"{M}/text_encoders", exist_ok=True)
os.makedirs(f"{M}/vae", exist_ok=True)
os.makedirs(f"{M}/clip", exist_ok=True)

HF = "https://huggingface.co/Kijai/CogVideoX_comfy/resolve/main"

files = {
    # CogVideoX-5B transformer INT8 (~5 GB)
    f"{M}/diffusion_models/cogvideox_5b_I2V_fp8_e4m3fn.safetensors":
        f"{HF}/cogvideox_5b_I2V_fp8_e4m3fn.safetensors",
    # T5 XXL text encoder FP8 (~4.9 GB)
    f"{M}/text_encoders/t5xxl_fp8_e4m3fn.safetensors":
        "https://huggingface.co/mcmonkey/google_t5-v1_1-xxl_encoderonly/resolve/main/t5xxl_fp8_e4m3fn.safetensors",
    # CogVideoX VAE (~400 MB)
    f"{M}/vae/cogvideox_vae.safetensors":
        f"{HF}/cogvideox_vae.safetensors",
}

for dest, url in files.items():
    if os.path.exists(dest):
        print(f"Already exists: {os.path.basename(dest)}")
    else:
        print(f"\nDownloading {os.path.basename(dest)}...")
        !wget -q --show-progress -O "{dest}" "{url}"

print("\nAll models ready!")
!du -sh {M}/diffusion_models/* {M}/text_encoders/* {M}/vae/*

In [ ]:
#@title 4. Download Workflow from Gist
import os

GIST = "ea1ab4a53e1be1b8401e5031bcdae4f3"
RAW = f"https://gist.githubusercontent.com/russiankendricklamar/{GIST}/raw"
WF_DIR = "/content/ComfyUI/user/default/workflows"
os.makedirs(WF_DIR, exist_ok=True)

!wget -q -O "{WF_DIR}/video_cogvideo_t2v.json" "{RAW}/video_cogvideo_t2v.json"
print("Workflow downloaded!")

In [ ]:
#@title 5. Upload Media (images for I2V mode)
from google.colab import files
import os

INPUT_DIR = "/content/ComfyUI/input"
os.makedirs(INPUT_DIR, exist_ok=True)

uploaded = files.upload()
for name, data in uploaded.items():
    path = os.path.join(INPUT_DIR, name)
    with open(path, "wb") as f:
        f.write(data)
    print(f"Saved: {path}")

print(f"\nFiles in input/: {os.listdir(INPUT_DIR)}")

In [ ]:
#@title 6. Launch ComfyUI
import subprocess, time, re

# Install cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# Start ComfyUI
comfy = subprocess.Popen(
    ["python", "main.py", "--listen", "0.0.0.0", "--port", "8188"],
    cwd="/content/ComfyUI",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)
print("Starting ComfyUI... (30s)")
time.sleep(30)

# Cloudflare tunnel
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8188"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)
time.sleep(5)

url = None
for _ in range(20):
    line = tunnel.stdout.readline().decode()
    match = re.search(r"https://[\w-]+\.trycloudflare\.com", line)
    if match:
        url = match.group(0)
        break

if url:
    print(f"\n{'='*60}")
    print(f"ComfyUI ready! Open: {url}")
    print(f"{'='*60}")
    print("\nLoad workflow: Menu -> Load -> video_cogvideo_t2v")
else:
    print("Tunnel URL not found. ComfyUI running on port 8188.")

In [ ]:
#@title 7. Save Outputs to Google Drive
from google.colab import drive
import shutil, glob, os

drive.mount('/content/drive')

src = "/content/ComfyUI/output"
dst = "/content/drive/MyDrive/ComfyUI_Output/cogvideo"
os.makedirs(dst, exist_ok=True)

videos = glob.glob(f"{src}/*.mp4") + glob.glob(f"{src}/*.png")
if not videos:
    print("No outputs found yet. Generate some videos first!")
else:
    for v in videos:
        shutil.copy2(v, dst)
        print(f"Copied: {os.path.basename(v)}")
    print(f"\n{len(videos)} files saved to {dst}")

---
## Usage Guide

### video_cogvideo_t2v — Text-to-Video
1. Open the tunnel URL
2. Load **video_cogvideo_t2v** workflow
3. Edit the text prompt
4. Click **Queue Prompt**

### Resolution & Duration
CogVideoX-5B generates at **720x480** (or 480x720 portrait) by default.
- 49 frames = ~6s at 8fps
- Keep frames at 49 for T4 to avoid OOM

### Prompt Tips
CogVideoX responds well to:
- **Scene descriptions**: "A serene lake at sunset, mountains in the background, gentle ripples"
- **Action sequences**: "A cat jumping from a shelf, slow motion, soft lighting"
- **Abstract**: "Flowing liquid gold forming into geometric shapes, dark background"

### I2V Mode
The workflow also supports Image-to-Video:
1. Upload an image in cell 5
2. In ComfyUI, connect a LoadImage node to the CogVideo sampler
3. The model will animate your image

### Comparison with Wan 2.2
- CogVideoX: smoother motion, more cinematic, lower resolution
- Wan 2.2: sharper details, more realistic, higher resolution
- Use both for variety in your content!